# Milestone 3, Task 5: Live Demo

Runs the fine-tuned YOLOv8n celebrity detector on held-out test images or on an image you provide.

* **Model and thresholds:** the checkpoint and operating point selected on the validation split by the
  hyperparameter sweep (`results/milestone3/sweep/best_model.pt` and `operating_point.json`, Task 3).
* **Part A** draws the detections on test images inside the notebook (no extra packages).
* **Part B** opens an interactive app in the notebook: upload an image or click a test example, and change the
  confidence threshold with a slider.

Each face is boxed in its identity color and labelled with the predicted identity and confidence.

#### 0. Environment setup (local Jupyter or Google Colab)
On Colab it mounts Drive, locally it finds the repo root.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rasmussenj03/IE7615-Group3-Project1.git"
COLAB_USE_DRIVE = True  # False -> clone into /content (outputs are lost when the runtime ends)
COLAB_REPO_DIR = ("/content/drive/MyDrive/IE7615-Group3-Project1" if COLAB_USE_DRIVE
                  else "/content/IE7615-Group3-Project1")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9", "gradio>=4.44"], check=True)
    if COLAB_USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path(COLAB_REPO_DIR, "src").is_dir():
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone = subprocess.run(["git", "clone", "-q", REPO_URL.replace("https://", f"https://{token}@"),
                                COLAB_REPO_DIR], capture_output=True)
        if clone.returncode != 0:
            raise RuntimeError("git clone failed - check the GITHUB_TOKEN secret and repo access")
        subprocess.run(["git", "-C", COLAB_REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    os.chdir(COLAB_REPO_DIR)
else:
    root = Path.cwd()
    while not (root / "results" / "milestone3" / "sweep" / "operating_point.json").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

PROJECT_ROOT = Path.cwd()

#### 1. Install requirements
Installs the packages in `requirements.txt` into this notebook's Python (runs after setup, which finds the repo).
If it reports that packages were updated, restart the kernel and run all cells again.

In [2]:
%pip install -q -r requirements.txt

#### 2. Load the model
Loads the fine-tuned checkpoint with the operating point from the sweep (confidence, NMS IoU, image size).

In [3]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

# Model and operating point selected by the hyperparameter sweep (Task 3)
OP = json.loads((PROJECT_ROOT / "results/milestone3/sweep/operating_point.json").read_text())
model = YOLO(str(PROJECT_ROOT / OP["model"]))
TEST_DIR = PROJECT_ROOT / "data/detection/images/test"
print("Operating point:", OP)


def detect(image, conf=OP["conf"]):
    """Runs the detector on one image; returns the annotated image (RGB) and one row per detected face."""
    result = model.predict(image, conf=conf, iou=OP["nms_iou"], imgsz=OP["imgsz"],
                           agnostic_nms=OP["agnostic_nms"], verbose=False)[0]
    faces = pd.DataFrame([[result.names[int(c)], round(float(s), 3), [round(v) for v in box]]
                          for c, s, box in zip(result.boxes.cls, result.boxes.conf, result.boxes.xyxy.tolist())],
                         columns=["identity", "confidence", "box (x1, y1, x2, y2)"])
    return result.plot()[:, :, ::-1], faces

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Julia\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Operating point: {'model': 'results/milestone3/sweep/best_model.pt', 'training_run': 'aug_light', 'imgsz': 640, 'conf': 0.8, 'nms_iou': 0.7, 'agnostic_nms': False, 'selected_on': 'validation F1'}


#### 3. Launch the app
Pick a test image from the list, drag and drop or upload an image, or take a photo with the webcam; the detections
update when the image or the confidence threshold changes. Run the next cell to stop the app.

In [6]:
import gradio as gr

TEST_IMAGES = sorted(p.name for p in TEST_DIR.glob("*.jpg"))


def run(image, conf):
    """Detection for the app; clears the outputs when no image is selected."""
    return detect(image, conf) if image else (None, None)


with gr.Blocks(title="IE7615 Group 3 detection demo") as app:
    gr.Markdown(f"## IE7615 Group 3: celebrity detection with fine-tuned YOLOv8n\n"
                f"Checkpoint `{OP['model']}` (run `{OP['training_run']}`), NMS IoU {OP['nms_iou']}, "
                f"image size {OP['imgsz']}. Pick one of the held-out test images, "
                f"drag and drop or upload an image, or take a webcam photo.")
    with gr.Row():
        with gr.Column():
            pick = gr.Dropdown(TEST_IMAGES, value="test_0001.jpg", label="Test image")
            image = gr.Image(str(TEST_DIR / "test_0001.jpg"), type="filepath", sources=["upload", "webcam"],
                             label="Input image (drag and drop, upload or webcam)")
            conf = gr.Slider(0.05, 0.95, value=OP["conf"], step=0.05, label="Confidence threshold")
            clear = gr.ClearButton(value="Clear")
        with gr.Column():
            detections = gr.Image(label="Detections")
            faces = gr.Dataframe(label="Faces")
    
    clear.add([pick, image, detections, faces])
    pick.input(lambda name: str(TEST_DIR / name), pick, image)
    image.change(run, [image, conf], [detections, faces])
    conf.release(run, [image, conf], [detections, faces])
    app.load(run, [image, conf], [detections, faces])

app.launch(share=IN_COLAB)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
app.close()  # stops the app

Closing server running on port: 7860
